# Study B Controllability V2 Analysis

Arm-aware notebook for the controllability `v2` path.

This notebook reads the same-case three-arm results:
- `spontaneous`
- `generic_control`
- `explicit_control`

Primary study view: `CHR / Controlled Hallucination Rate`

Notes:
- The arm applies to both control and injected variants in the same case set.


In [ ]:
import sys
from pathlib import Path

for candidate_root in [Path.cwd(), Path.cwd().parent]:
    src_dir = candidate_root / "src"
    if src_dir.exists() and str(src_dir.resolve()) not in sys.path:
        sys.path.insert(0, str(src_dir.resolve()))

import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

RESULTS_DIR = next(
    (p for p in [Path("results"), Path("../results"), Path("../../results")] if p.exists()),
    Path("results"),
)
print(f"Using RESULTS_DIR: {RESULTS_DIR.resolve()}")


In [ ]:
STUDY_KEY = "B"
PRIMARY_LABEL = "CHR / Controlled Hallucination Rate"
TASK_METRIC_COLUMNS = ['task_sycophancy_probability', 'task_flip_rate', 'task_accuracy_injected']


In [ ]:
def load_controllability_v2_study_payload(study_key: str):
    study_file_map = {
        "A": "ctrl_v2_study_a_results.json",
        "A_bias": "ctrl_v2_study_a_bias_results.json",
        "B": "ctrl_v2_study_b_results.json",
        "B_multi_turn": "ctrl_v2_study_b_multi_turn_results.json",
        "C": "ctrl_v2_study_c_results.json",
    }
    target_file = study_file_map[study_key]

    arm_rows = []
    delta_rows = []
    summary_rows = []

    if not RESULTS_DIR.exists():
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    for model_dir in sorted(path for path in RESULTS_DIR.iterdir() if path.is_dir()):
        study_path = model_dir / target_file
        if study_path.exists():
            payload = json.loads(study_path.read_text(encoding="utf-8"))
            exclusions = payload.get("exclusions") or {}
            notes = payload.get("notes") or []
            for arm_name, arm_payload in sorted((payload.get("arms") or {}).items()):
                primary_metric = arm_payload.get("primary_metric") or {}
                row = {
                    "model": payload.get("model", model_dir.name),
                    "study": payload.get("study", study_key),
                    "arm": arm_name,
                    "primary_metric_name": primary_metric.get("metric_name"),
                    "primary_metric_value": primary_metric.get("value"),
                    "primary_ci_lower": primary_metric.get("ci_lower"),
                    "primary_ci_upper": primary_metric.get("ci_upper"),
                    "notes": " | ".join(arm_payload.get("notes") or []),
                }
                for key, value in (arm_payload.get("task_metrics") or {}).items():
                    row[f"task_{key}"] = value
                for key, value in (arm_payload.get("counts") or {}).items():
                    row[f"count_{key}"] = value
                for key, value in exclusions.items():
                    row[f"exclusion_{key}"] = value
                arm_rows.append(row)

            for delta in payload.get("pairwise_deltas", []):
                row = {
                    "model": payload.get("model", model_dir.name),
                    "study": payload.get("study", study_key),
                    "from_arm": delta.get("from_arm"),
                    "to_arm": delta.get("to_arm"),
                    "n_pairs": delta.get("n_pairs"),
                    "notes": " | ".join(delta.get("notes") or []),
                }
                for key, value in (delta.get("metrics") or {}).items():
                    row[f"delta_{key}"] = value
                delta_rows.append(row)

        summary_path = model_dir / "controllability_v2_summary.json"
        if summary_path.exists():
            summary_payload = json.loads(summary_path.read_text(encoding="utf-8"))
            study_payload = (summary_payload.get("studies") or {}).get(study_key)
            if study_payload:
                summary_rows.append(
                    {
                        "model": summary_payload.get("model", model_dir.name),
                        "study": study_key,
                        "arm_count": len(study_payload.get("arms") or {}),
                    }
                )

    return pd.DataFrame(arm_rows), pd.DataFrame(delta_rows), pd.DataFrame(summary_rows)


In [ ]:
arm_df, delta_df, summary_df = load_controllability_v2_study_payload(STUDY_KEY)
if arm_df.empty:
    print(f"No controllability v2 results found yet for {STUDY_KEY}.")
else:
    display(arm_df.sort_values(["arm", "primary_metric_value"], ascending=[True, False]).reset_index(drop=True))


## Primary Metric by Arm


In [ ]:
if arm_df.empty:
    print("Skipping primary metric plot - no v2 study results found.")
else:
    plot_df = arm_df.sort_values(["arm", "model"]).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(16, 7))
    sns.barplot(data=plot_df, x="model", y="primary_metric_value", hue="arm", ax=ax)
    ax.set_title(f"{STUDY_KEY} primary controllability metric by arm", fontsize=14, fontweight="bold")
    ax.set_xlabel("Model")
    ax.set_ylabel("Primary metric value")
    plt.xticks(rotation=45, ha="right")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()


## Task Metrics by Arm


In [ ]:
if arm_df.empty:
    print("Skipping task-metric view - no v2 study results found.")
else:
    available_columns = [col for col in TASK_METRIC_COLUMNS if col in arm_df.columns and arm_df[col].notna().any()]
    if not available_columns:
        print("No requested task metrics are populated yet.")
    else:
        metric_df = arm_df[["model", "arm"] + available_columns].copy()
        long_df = metric_df.melt(id_vars=["model", "arm"], value_vars=available_columns, var_name="metric", value_name="value").dropna(subset=["value"])
        display(long_df.sort_values(["metric", "arm", "model"]).reset_index(drop=True))
        fig, ax = plt.subplots(figsize=(16, 7))
        sns.barplot(data=long_df, x="metric", y="value", hue="arm", ax=ax)
        ax.set_title(f"{STUDY_KEY} task metrics by arm", fontsize=14, fontweight="bold")
        ax.set_xlabel("Metric")
        ax.set_ylabel("Value")
        plt.xticks(rotation=35, ha="right")
        ax.grid(axis="y", alpha=0.3)
        plt.tight_layout()
        plt.show()


## Pairwise Deltas


In [ ]:
if delta_df.empty:
    print("No pairwise arm deltas are available yet.")
else:
    display(delta_df.sort_values(["from_arm", "to_arm", "model"]).reset_index(drop=True))


## Coverage Summary


In [ ]:
if summary_df.empty:
    print("No v2 summary rows were found for this study yet.")
else:
    display(summary_df.sort_values("model").reset_index(drop=True))
